# ETL Pipeline: Knapsack Problem Instance Processing

Paola A. Castillo-Gutiérrez

## Overview

This notebook implements an **Extract-Transform-Load (ETL)** pipeline for processing synthetic 0/1 Knapsack Problem (KP) instances generated via the evolutionary-based approach described in Plata-González et al. (2019).

The pipeline performs three stages:

1. **Extract**: Read raw `.kp` instance files from organized subdirectories.
2. **Transform**: Parse each file's structure (metadata + item data), extract instance-level features, and assign labels derived from the filename convention.
3. **Load**: Produce two consolidated CSV outputs (a unified item-level dataset and an instance-level feature dataset).

## Source File Format (`.kp`)

Each `.kp` file encodes a single 0/1 KP instance with the following structure:

| Row | Column 1 | Column 2 |
|-----|----------|----------|
| 1 (header) | `n` (number of items) | `C` (knapsack capacity) |
| 2 to `n+1` | `w_j` (weight of item `j`) | `p_j` (profit of item `j`) |

The files are organized in subdirectories by experimental set (e.g., easy/hard instances per heuristic).

## Instance Features (Plata-González et al., 2019)

Seven normalized features characterize each instance, all bounded to $[0, 1]$ and rounded to 3 decimal places:

| Feature | Symbol | Formula |
|---------|--------|---------|
| Mean weight (normalized) | $\bar{w}$ | $\text{mean}(w) \;/\; \max(w)$ |
| Median weight (normalized) | $\tilde{w}$ | $\text{median}(w) \;/\; \max(w)$ |
| Std. dev. of weights (normalized) | $\sigma_w$ | $\text{std}(w, \text{ddof}=1) \;/\; \max(w)$ |
| Mean profit (normalized) | $\bar{p}$ | $\text{mean}(p) \;/\; \max(p)$ |
| Median profit (normalized) | $\tilde{p}$ | $\text{median}(p) \;/\; \max(p)$ |
| Std. dev. of profits (normalized) | $\sigma_p$ | $\text{std}(p, \text{ddof}=1) \;/\; \max(p)$ |
| Weight-profit correlation (shifted) | $r$ | $\text{corr}(w, p) \;/\; 2 + 0.5$ |

The standard deviation uses the sample estimator (`ddof=1`), consistent with the paper's worked example (Section 2.2, p. 12713).

The correlation transformation maps Pearson's $r \in [-1, 1]$ to $[0, 1]$.

Additionally, four raw range descriptors are included per instance (not normalized): `weight_min`, `weight_max`, `profit_min`, `profit_max`.

### References (APA 7)

1. Plata-González, L. F., Amaya, I., Ortiz-Bayliss, J. C., Conant-Pablos, S. E.,
   Terashima-Marín, H., & Coello Coello, C. A. (2019). Evolutionary-based tailoring
   of synthetic instances for the Knapsack problem. *Soft Computing*, *23*, 12711–12728.
   https://doi.org/10.1007/s00500-019-03822-w

2. Pisinger, D. (2005). Where are the hard knapsack problems? *Computers & Operations
   Research*, *32*(9), 2271–2284. https://doi.org/10.1016/j.cor.2004.03.002

In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from dataclasses import dataclass

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Configuration

In [3]:
BASE_DIR = Path('/content/drive/MyDrive/KP Instances 2026')
OUTPUT_DIR = Path('/content/drive/MyDrive/KP_Instances_Processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Domain Model

In [8]:
@dataclass
class KPInstance:
    filename: str
    set_name: str
    n_items: int
    capacity: int
    weights: np.ndarray
    profits: np.ndarray
    dominant_heuristic: str
    instance_type: str

    @staticmethod
    def from_file(filepath: Path, set_name: str) -> 'KPInstance':
        lines = filepath.read_text().strip().splitlines()
        tokens = [[t.strip(',') for t in line.split()] for line in lines if line.strip()]

        n_items = int(tokens[0][0])
        capacity = int(tokens[0][1])

        items = np.array(tokens[1:], dtype=float)
        weights = items[:, 0]
        profits = items[:, 1]

        if len(weights) != n_items:
            raise ValueError(
                f"{filepath.name}: header declares {n_items} items "
                f"but file contains {len(weights)} item rows"
            )

        dominant_heuristic, instance_type = KPInstance._parse_label(filepath.stem)

        return KPInstance(
            filename=filepath.name,
            set_name=set_name,
            n_items=n_items,
            capacity=capacity,
            weights=weights,
            profits=profits,
            dominant_heuristic=dominant_heuristic,
            instance_type=instance_type,
        )

    @staticmethod
    def _parse_label(stem: str) -> tuple:
        stem_upper = stem.upper()
        if '_EASY' in stem_upper:
            tag = '_EASY'
        elif '_HARD' in stem_upper:
            tag = '_HARD'
        else:
            return ('unknown', 'unknown')

        idx = stem_upper.index(tag)
        heuristic = stem[:idx]
        instance_type = tag.strip('_').lower()
        return (heuristic, instance_type)

In [9]:
class KPFeatureExtractor:

    DECIMALS = 3

    @classmethod
    def compute(cls, instance: KPInstance) -> dict:
        w = instance.weights.astype(float)
        p = instance.profits.astype(float)
        max_w = w.max()
        max_p = p.max()

        if max_w == 0 or max_p == 0:
            return cls._nan_features()

        features = {
            'w_mean':   round(w.mean() / max_w, cls.DECIMALS),
            'w_median': round(float(np.median(w)) / max_w, cls.DECIMALS),
            'w_std':    round(w.std(ddof=1) / max_w, cls.DECIMALS),
            'p_mean':   round(p.mean() / max_p, cls.DECIMALS),
            'p_median': round(float(np.median(p)) / max_p, cls.DECIMALS),
            'p_std':    round(p.std(ddof=1) / max_p, cls.DECIMALS),
            'wp_corr_raw': cls._shifted_correlation(w, p)[0],
            'wp_corr':  cls._shifted_correlation(w, p)[1],
            'weight_min': int(w.min()),
            'weight_max': int(max_w),
            'profit_min': int(p.min()),
            'profit_max': int(max_p),
            'capacity_ratio': round(instance.capacity / w.sum(), cls.DECIMALS),
        }

        cls._validate_bounds(features, instance.filename)
        return features

    @classmethod
    def _shifted_correlation(cls, w: np.ndarray, p: np.ndarray) -> tuple:
        if w.std() == 0 or p.std() == 0:
            return (0.0, 0.5)
        r = float(np.corrcoef(w, p)[0, 1])
        return (round(r, cls.DECIMALS), round(r / 2 + 0.5, cls.DECIMALS))

    @staticmethod
    def _nan_features() -> dict:
        norm_keys = ['w_mean', 'w_median', 'w_std', 'p_mean', 'p_median', 'p_std', 'wp_corr_raw', 'wp_corr', 'capacity_ratio']
        raw_keys = ['weight_min', 'weight_max', 'profit_min', 'profit_max']
        result = {k: np.nan for k in norm_keys}
        result.update({k: np.nan for k in raw_keys})
        return result

    @staticmethod
    def _validate_bounds(features: dict, filename: str):
        normalized_keys = ['w_mean', 'w_median', 'w_std', 'p_mean', 'p_median', 'p_std', 'wp_corr']
        for key in normalized_keys:
            val = features[key]
            if not np.isnan(val) and not (0.0 <= val <= 1.0):
                print(f"  Warning: {filename} -> {key}={val} is outside [0, 1]")

## 2. ETL Pipeline

In [10]:
class KPPipeline:

    VALID_EXTENSIONS = {'.kp', '.txt', '.csv'}

    FEATURE_COLUMN_ORDER = [
        'filename', 'set', 'dominant_heuristic', 'instance_type',
        'n_items', 'capacity',
        'weight_min', 'weight_max', 'profit_min', 'profit_max',
        'w_mean', 'w_median', 'w_std',
        'p_mean', 'p_median', 'p_std',
        'wp_corr_raw', 'wp_corr',
        'capacity_ratio',
    ]

    ITEM_COLUMN_ORDER = [
        'filename', 'set', 'dominant_heuristic', 'instance_type',
        'weight', 'profit',
    ]

    def __init__(self, base_dir: Path):
        self.base_dir = base_dir
        self.instances: list[KPInstance] = []
        self.errors: list[str] = []

    def extract(self) -> 'KPPipeline':
        subfolders = sorted(
            [d for d in self.base_dir.iterdir() if d.is_dir()]
        )
        print(f"Found {len(subfolders)} subfolders: {[d.name for d in subfolders]}")

        for subfolder in subfolders:
            files = sorted(
                [f for f in subfolder.iterdir()
                 if f.is_file() and f.suffix in self.VALID_EXTENSIONS]
            )
            count = 0
            for filepath in files:
                try:
                    inst = KPInstance.from_file(filepath, subfolder.name)
                    self.instances.append(inst)
                    count += 1
                except Exception as e:
                    self.errors.append(f"{filepath}: {e}")

            print(f"  {subfolder.name}: {count} instances loaded")

        print(f"\nTotal: {len(self.instances)} instances, {len(self.errors)} errors")
        if self.errors:
            for err in self.errors[:10]:
                print(f"  ERROR: {err}")
        return self

    def transform(self) -> tuple[pd.DataFrame, pd.DataFrame]:
        item_rows = []
        feature_rows = []

        for inst in self.instances:
            for w, p in zip(inst.weights, inst.profits):
                item_rows.append({
                    'filename': inst.filename,
                    'set': inst.set_name,
                    'dominant_heuristic': inst.dominant_heuristic,
                    'instance_type': inst.instance_type,
                    'weight': int(w),
                    'profit': int(p),
                })

            features = KPFeatureExtractor.compute(inst)
            features.update({
                'filename': inst.filename,
                'set': inst.set_name,
                'dominant_heuristic': inst.dominant_heuristic,
                'instance_type': inst.instance_type,
                'n_items': inst.n_items,
                'capacity': inst.capacity,
            })
            feature_rows.append(features)

        df_items = pd.DataFrame(item_rows)[self.ITEM_COLUMN_ORDER]
        df_features = pd.DataFrame(feature_rows)[self.FEATURE_COLUMN_ORDER]

        return df_items, df_features

    def load(self, df_items: pd.DataFrame, df_features: pd.DataFrame,
             output_dir: Path) -> None:
        items_path = output_dir / 'kp_instances_unified.csv'
        features_path = output_dir / 'kp_features.csv'

        df_items.to_csv(items_path, index=False)
        df_features.to_csv(features_path, index=False)

        print(f"Saved {len(df_items)} item rows -> {items_path}")
        print(f"Saved {len(df_features)} instance features -> {features_path}")

    def run(self, output_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
        self.extract()
        df_items, df_features = self.transform()
        self.load(df_items, df_features, output_dir)
        return df_items, df_features

## 3. Execute Pipeline

In [11]:
pipeline = KPPipeline(BASE_DIR)
df_items, df_features = pipeline.run(OUTPUT_DIR)

Found 16 subfolders: ['Set-100-128', 'Set-100-256', 'Set-100-32', 'Set-100-64', 'Set-200-128', 'Set-200-256', 'Set-200-32', 'Set-200-64', 'Set-25-128', 'Set-25-256', 'Set-25-32', 'Set-25-64', 'Set-50-128', 'Set-50-256', 'Set-50-32', 'Set-50-64']
  Set-100-128: 400 instances loaded
  Set-100-256: 400 instances loaded
  Set-100-32: 400 instances loaded
  Set-100-64: 400 instances loaded
  Set-200-128: 400 instances loaded
  Set-200-256: 400 instances loaded
  Set-200-32: 400 instances loaded
  Set-200-64: 400 instances loaded
  Set-25-128: 400 instances loaded
  Set-25-256: 400 instances loaded
  Set-25-32: 400 instances loaded
  Set-25-64: 400 instances loaded
  Set-50-128: 400 instances loaded
  Set-50-256: 400 instances loaded
  Set-50-32: 400 instances loaded
  Set-50-64: 400 instances loaded

Total: 6400 instances, 0 errors
Saved 600000 item rows -> /content/drive/MyDrive/KP_Instances_Processed/kp_instances_unified.csv
Saved 6400 instance features -> /content/drive/MyDrive/KP_Instan

## 4. Data Inspection

In [12]:
print("Item-level dataset")
print(f"  Shape: {df_items.shape}")
print(f"  Columns: {list(df_items.columns)}")
print(f"  Heuristics: {df_items['dominant_heuristic'].unique()}")
print(f"  Instance types: {df_items['instance_type'].unique()}")
df_items.head(10)

Item-level dataset
  Shape: (600000, 6)
  Columns: ['filename', 'set', 'dominant_heuristic', 'instance_type', 'weight', 'profit']
  Heuristics: ['DEF' 'MAXPW' 'MAXP' 'MINW']
  Instance types: ['easy']


,filename,set,dominant_heuristic,instance_type,weight,profit
0,DEF_EASY_100_000.kp,Set-100-128,DEF,easy,15,84
1,DEF_EASY_100_000.kp,Set-100-128,DEF,easy,23,99
2,DEF_EASY_100_000.kp,Set-100-128,DEF,easy,11,90
3,DEF_EASY_100_000.kp,Set-100-128,DEF,easy,8,89
4,DEF_EASY_100_000.kp,Set-100-128,DEF,easy,5,96
5,DEF_EASY_100_000.kp,Set-100-128,DEF,easy,10,73
6,DEF_EASY_100_000.kp,Set-100-128,DEF,easy,13,86
7,DEF_EASY_100_000.kp,Set-100-128,DEF,easy,53,95
8,DEF_EASY_100_000.kp,Set-100-128,DEF,easy,17,82
9,DEF_EASY_100_000.kp,Set-100-128,DEF,easy,19,100


In [13]:
print("Feature-level dataset")
print(f"  Shape: {df_features.shape}")
print(f"  Columns: {list(df_features.columns)}")
print(f"  Capacity range: [{df_features['capacity'].min()}, {df_features['capacity'].max()}]")
print(f"  n_items range: [{df_features['n_items'].min()}, {df_features['n_items'].max()}]")
df_features.describe()

Feature-level dataset
  Shape: (6400, 19)
  Columns: ['filename', 'set', 'dominant_heuristic', 'instance_type', 'n_items', 'capacity', 'weight_min', 'weight_max', 'profit_min', 'profit_max', 'w_mean', 'w_median', 'w_std', 'p_mean', 'p_median', 'p_std', 'wp_corr_raw', 'wp_corr', 'capacity_ratio']
  Capacity range: [32, 256]
  n_items range: [25, 200]


,n_items,capacity,weight_min,weight_max,profit_min,profit_max,w_mean,w_median,w_std,p_mean,p_median,p_std,wp_corr_raw,wp_corr,capacity_ratio
count,6400.00000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000
mean,93.75000,120.000000,1.543438,59.649219,2.314844,99.767031,0.555015,0.554434,0.278694,0.526943,0.545670,0.318574,-0.043144,0.478426,0.071550
std,67.02902,85.797146,1.366096,42.556434,2.828830,0.951853,0.095961,0.154013,0.026926,0.054101,0.116585,0.050698,0.159003,0.079506,0.057465
min,25.00000,32.000000,1.000000,14.000000,1.000000,89.000000,0.271000,0.181000,0.193000,0.343000,0.210000,0.172000,-0.607000,0.197000,0.015000
25%,43.75000,56.000000,1.000000,25.750000,1.000000,100.000000,0.492000,0.469000,0.258000,0.490000,0.465000,0.287000,-0.146250,0.427000,0.027000
50%,75.00000,96.000000,1.000000,44.000000,1.000000,100.000000,0.579000,0.594000,0.279000,0.519000,0.530000,0.304000,-0.037000,0.481000,0.054500
75%,125.00000,160.000000,2.000000,74.500000,3.000000,100.000000,0.627000,0.660000,0.298000,0.557000,0.600000,0.337000,0.066000,0.533000,0.114000
max,200.00000,256.000000,23.000000,128.000000,37.000000,100.000000,0.768000,0.875000,0.379000,0.768000,0.990000,0.485000,0.582000,0.791000,0.296000


In [14]:
nan_count = df_features[['w_mean', 'w_median', 'w_std', 'p_mean', 'p_median', 'p_std', 'wp_corr']].isna().sum()
if nan_count.any():
    print("NaN counts per feature:")
    print(nan_count[nan_count > 0])
else:
    print("No NaN values in any feature column.")

No NaN values in any feature column.


## 5. Feature Verification Against Paper Example

Section 2.2 (p. 12713) of Plata-González et al. (2019) provides a worked example:

> "Consider a subset of remaining items whose weights are (2, 2, 3, 4) and whose profits are (10, 5, 6, 15), respectively. Thus, each of the aforementioned features (in order) will take the values of (0.69, 0.63, 0.24, 0.60, 0.53, 0.30, 0.69)."

### Analysis of the first six features ($\bar{w}, \tilde{w}, \sigma_w, \bar{p}, \tilde{p}, \sigma_p$)

These six values are reproducible with `ddof=1` (sample standard deviation) when rounded to 2 decimal places. Using `ddof=0` (population standard deviation) yields $\sigma_w = 0.21$ and $\sigma_p = 0.26$, which do not match the paper's 0.24 and 0.30.

### Analysis of the seventh feature ($r = 0.69$)

The paper defines $r$ as: "weight-profit correlation, divided by two and shifted upwards by 0.5." For the given data:

| Metric | Raw value | After transformation ($\cdot / 2 + 0.5$) |
|--------|-----------|-------------------------------------------|
| Pearson $r$ | 0.689 | 0.845 |
| Spearman $\rho$ | 0.632 | 0.816 |

The paper's reported value of 0.69 matches the raw Pearson correlation without applying the transformation.

### Resolution

Both quantities are now stored explicitly:

| Column | Formula | Range | Interpretation |
|--------|---------|-------|----------------|
| `wp_corr_raw` | $r = \text{corr}(w, p)$ | $[-1, 1]$ | Raw Pearson; reproduces the paper's worked example value of 0.69 |
| `wp_corr` | $r / 2 + 0.5$ | $[0, 1]$ | Normalized per the paper's formula; consistent with the $[0,1]$ bound of all other features |

### Additional feature: `capacity_ratio`

$$\text{capacity\_ratio} = \frac{C}{\sum_{j=1}^{n} w_j}$$

This ratio measures the relative tightness of the knapsack constraint. A value close to 0 indicates a highly constrained instance; a value $\geq 1$ means the capacity is non-binding (all items can be packed). This feature is not part of Plata-González et al. (2019) but is a standard descriptor in the KP literature for characterizing instance difficulty (Pisinger, 2005).


In [15]:
test_instance = KPInstance(
    filename='verification_example',
    set_name='test',
    n_items=4,
    capacity=10,
    weights=np.array([2, 2, 3, 4], dtype=float),
    profits=np.array([10, 5, 6, 15], dtype=float),
    dominant_heuristic='test',
    instance_type='test',
)

result = KPFeatureExtractor.compute(test_instance)

paper_values = {
    'w_mean':   (0.69, ''),
    'w_median': (0.63, ''),
    'w_std':    (0.24, ''),
    'p_mean':   (0.60, ''),
    'p_median': (0.53, ''),
    'p_std':    (0.30, ''),
    'wp_corr_raw': (0.69, 'raw Pearson r -- matches paper example directly'),
    'wp_corr':     (0.845, 'r/2+0.5 -- paper formula; example inconsistency confirmed'),
    'capacity_ratio': (None, 'C / sum(w) = 10/11 ~ 0.909; not in paper'),
}

print(f"{'Feature':<16} {'Computed':>10} {'Paper':>10} {'Match':>6}  Notes")
print('-' * 80)
for key, (expected, note) in paper_values.items():
    computed = result[key]
    if expected is None:
        match_str = 'N/A'
    else:
        match_str = 'Y' if abs(computed - expected) < 0.015 else 'N'
    expected_str = f'{expected:.3f}' if expected is not None else '  ---'
    print(f"{key:<16} {computed:>10.3f} {expected_str:>10} {match_str:>6}  {note}")

print(f"\nRaw range descriptors: weight=[{result['weight_min']}, {result['weight_max']}], "
      f"profit=[{result['profit_min']}, {result['profit_max']}]")
print(f"\nCorrelation reconciliation:")
print(f"  wp_corr_raw = {result['wp_corr_raw']:.3f}  -> matches paper's reported 0.69 (raw Pearson)")
print(f"  wp_corr     = {result['wp_corr']:.3f}  -> correct application of paper's formula (r/2+0.5)")
print(f"  Both are now stored. ")


Feature            Computed      Paper  Match  Notes
--------------------------------------------------------------------------------
w_mean                0.688      0.690      Y  
w_median              0.625      0.630      Y  
w_std                 0.239      0.240      Y  
p_mean                0.600      0.600      Y  
p_median              0.533      0.530      Y  
p_std                 0.303      0.300      Y  
wp_corr_raw           0.689      0.690      Y  raw Pearson r -- matches paper example directly
wp_corr               0.845      0.845      Y  r/2+0.5 -- paper formula; example inconsistency confirmed
capacity_ratio        0.909        ---    N/A  C / sum(w) = 10/11 ~ 0.909; not in paper

Raw range descriptors: weight=[2, 4], profit=[5, 15]

Correlation reconciliation:
  wp_corr_raw = 0.689  -> matches paper's reported 0.69 (raw Pearson)
  wp_corr     = 0.845  -> correct application of paper's formula (r/2+0.5)
  Both are now stored. 
